# 01 — EDA: how much do entities repeat?

The whole project rests on one assumption: fraudulent transactions **share** cards, devices, addresses and email domains with other transactions, and that sharing carries signal. This notebook checks that before we build anything.

1. Load the data and check coverage
2. How often is each entity shared?
3. Does fraud rate change with how widely an entity is shared?
4. Quick preview: does "guilt by association" predict fraud on later transactions?
5. Graph structure: components

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score

from hgfraud.data import load_train, add_entity_keys, entity_columns
from hgfraud.graph import build_graph, graph_summary, ENTITY_TYPES
from hgfraud.features import graph_features
from hgfraud.model import time_split

# Folder with train_transaction.csv and train_identity.csv.
# Either copy the CSVs into data/raw/, or point straight at where they are, e.g.
# DATA_DIR = Path(r"C:\Users\<you>\Downloads\ieee-fraud-detection")
DATA_DIR = Path("../data/raw")
NROWS = None          # e.g. 100_000 for a fast first run (full data needs ~4 GB RAM)

pd.set_option("display.float_format", "{:.4f}".format)
REPORTS = Path("../reports/figures"); REPORTS.mkdir(parents=True, exist_ok=True)

## 1. Load and check coverage
Only the columns needed for entity keys are loaded, to keep memory low.

In [ ]:
df = load_train(DATA_DIR, nrows=NROWS, usecols=entity_columns())
df = add_entity_keys(df)

days = (df["TransactionDT"].max() - df["TransactionDT"].min()) / 86400
print(f"{len(df):,} transactions over {days:.0f} days")
print(f"fraud rate: {df['isFraud'].mean():.2%}")
print(f"with identity record: {df['DeviceType'].notna().mean():.1%}")
df.head()

## 2. How often is each entity shared?
`shared %` = share of transactions whose entity appears in at least one other transaction.

In [ ]:
ENTITY_COLS = {**ENTITY_TYPES, "r_email": "recipient_email"}

rows = []
for col, name in ENTITY_COLS.items():
    key = df[col]
    deg = key.map(key.value_counts())
    rows.append({
        "entity": name,
        "missing %": key.isna().mean() * 100,
        "unique": key.nunique(),
        "shared %": (deg > 1).mean() * 100,
        "median degree": deg.median(),
        "max degree": deg.max(),
        "fraud rate (missing)": df.loc[key.isna(), "isFraud"].mean(),
        "fraud rate (present)": df.loc[key.notna(), "isFraud"].mean(),
    })
sharing = pd.DataFrame(rows).set_index("entity")
sharing

## 3. Fraud rate by how widely the entity is shared
If fraud rings reuse cards and devices, fraud should concentrate at some degree ranges rather than being flat.

In [ ]:
bins = [0, 1, 5, 20, 100, 1000, np.inf]
labels = ["1", "2-5", "6-20", "21-100", "101-1000", ">1000"]

fig, axes = plt.subplots(1, len(ENTITY_TYPES), figsize=(4 * len(ENTITY_TYPES), 3.5), sharey=True)
by_degree = {}
for ax, (col, name) in zip(axes, ENTITY_TYPES.items()):
    key = df[col]
    deg = key.map(key.value_counts())
    bucket = pd.cut(deg, bins=bins, labels=labels)
    t = df.groupby(bucket, observed=False)["isFraud"].agg(["mean", "size"])
    by_degree[name] = t
    ax.bar(t.index.astype(str), t["mean"], color="#4C72B0")
    ax.axhline(df["isFraud"].mean(), ls="--", c="grey", lw=1)
    ax.set_title(name); ax.set_xlabel("degree"); ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("fraud rate")
plt.tight_layout(); plt.savefig(REPORTS / "fraud_rate_by_degree.png", dpi=150); plt.show()

pd.concat(by_degree, names=["entity", "degree"])

Which individual cards and devices look like fraud hubs? (min. 5 transactions)

In [ ]:
for col, name in [("card_key", "card"), ("device_key", "device")]:
    g = df.groupby(col)["isFraud"].agg(n="size", fraud="sum", rate="mean")
    hubs = g[g["n"] >= 5].sort_values(["rate", "n"], ascending=False)
    print(f"\n{name}s with >=5 tx: {len(hubs):,};  with >=50% fraud: {(hubs['rate'] >= 0.5).sum():,}")
    display(hubs.head(10))

## 4. Preview: does guilt by association predict *future* fraud?
Fit nothing yet — just score the later 20% of transactions with each single graph feature, computed from training-period labels only, and compare PR-AUC with the base rate. A useful feature should be well above the base rate.

In [ ]:
train, test = time_split(df)
feats = graph_features(df, None, train_index=train.index)

y = test["isFraud"]
base = y.mean()
uni = {c: average_precision_score(y, feats.loc[test.index, c].fillna(0)) for c in feats.columns}
uni = pd.Series(uni, name="PR-AUC").sort_values(ascending=False).to_frame()
uni["lift vs base"] = uni["PR-AUC"] / base
print(f"base rate (random PR-AUC): {base:.4f}")
uni

## 5. Graph structure
Email domains like gmail.com and common addresses link almost everything into one giant component, which would make `component_size` meaningless. So we compare the full graph with one built from the more specific entities only (card + device).

In [ ]:
def component_sizes(G):
    sizes = []
    for comp in __import__("networkx").connected_components(G):
        n_tx = sum(1 for n in comp if n.startswith("tx:"))
        if n_tx:
            sizes.append(n_tx)
    return pd.Series(sorted(sizes, reverse=True))

G_full = build_graph(df)
G_specific = build_graph(df, {"card_key": "card", "device_key": "device"})

for name, G in [("full", G_full), ("card+device", G_specific)]:
    s = component_sizes(G)
    print(f"\n{name}: {len(s):,} components; largest = {s.iloc[0]:,} tx ({s.iloc[0]/len(df):.1%} of all)")
    display(graph_summary(G).to_frame("count").T)

In [ ]:
f = graph_features(df, G_specific, train_index=df.index)
bucket = pd.cut(f["component_size"], bins=[0, 1, 5, 20, 100, 1000, np.inf], labels=labels)
comp_tab = df.groupby(bucket, observed=False)["isFraud"].agg(fraud_rate="mean", n="size")

ax = comp_tab["fraud_rate"].plot.bar(color="#DD8452", figsize=(6, 3.5))
ax.axhline(df["isFraud"].mean(), ls="--", c="grey", lw=1)
ax.set_xlabel("component size (card+device graph, # transactions)"); ax.set_ylabel("fraud rate")
plt.tight_layout(); plt.savefig(REPORTS / "fraud_rate_by_component.png", dpi=150); plt.show()
comp_tab

## Takeaways
_Fill in after running:_

- Which entities are shared enough to be useful?
- Where does fraud concentrate by degree / component size?
- Which single graph features beat the base rate on future transactions?
- Which entity types should go into the final graph (and which create a useless giant component)?